# 03 Statistical Forecasting - SARIMA and SARIMAX

This notebook compares practical SARIMA and SARIMAX models against the Phase 2 baseline benchmark. The seasonal naive model is the benchmark to beat. No Prophet, deep learning or simulation models are built here.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src import statistical_models
from src.utils import FIGURES_DIR, TABLES_DIR

TARGET = "nd_mean"
TEST_START = "2025-01-01"
TEST_END = "2025-12-31"

## Load Daily Dataset and Baseline Results

In [ ]:
daily_df = statistical_models.load_daily_dataset()
baseline_comparison, baseline_forecasts = statistical_models.load_baseline_results()

print("Daily dataset shape:", daily_df.shape)
print("Baseline comparison available:", not baseline_comparison.empty)
display(daily_df.head())
display(baseline_comparison)

## Exogenous Features

In [ ]:
exog = statistical_models.select_exogenous_features(daily_df)
print("SARIMAX exogenous features used:")
print(list(exog.columns))

## Run Statistical Forecasting Pipeline

In [ ]:
comparison, forecasts = statistical_models.run_statistical_forecasting(
    target=TARGET,
    test_start=TEST_START,
    test_end=TEST_END,
)
display(comparison)
display(forecasts.head())

## Forecast Plots

In [ ]:
for figure_path in [
    FIGURES_DIR / "modelling" / "actual_vs_statistical_forecasts.png",
    FIGURES_DIR / "modelling" / "statistical_forecast_errors.png",
    FIGURES_DIR / "modelling" / "model_mape_comparison.png",
]:
    print(figure_path)
    if figure_path.exists():
        image = plt.imread(figure_path)
        plt.figure(figsize=(14, 6))
        plt.imshow(image)
        plt.axis("off")
        plt.show()

## Benchmark Interpretation

In [ ]:
seasonal_row = comparison.loc[comparison["model"] == "seasonal_naive"]
if seasonal_row.empty:
    print("Seasonal naive benchmark is not available. Re-run `python src/baseline_models.py --target nd_mean` first.")
else:
    seasonal_mae = seasonal_row["mae"].iloc[0]
    for model in ["sarima", "sarimax"]:
        model_mae = comparison.loc[comparison["model"] == model, "mae"].iloc[0]
        if model_mae < seasonal_mae:
            print(f"{model} beats seasonal_naive on MAE: {model_mae:.2f} vs {seasonal_mae:.2f}")
        else:
            print(f"{model} does not beat seasonal_naive on MAE: {model_mae:.2f} vs {seasonal_mae:.2f}")

SARIMA/SARIMAX should only be described as an improvement if they beat the seasonal naive benchmark on the same 2025 test period. If they do not beat it, seasonal naive remains the best model so far and later modelling should focus on understanding why the simpler seasonal structure is so competitive.